# Day 066 — Exercise 5: Image Pipeline

**What you'll build:** `image_pipeline(img, steps)` — apply a sequence of named operations to an image.

**Why it matters:** A steps-list design makes image pipelines data-driven — you can specify the sequence in JSON or a config file. This pattern is used by image processing APIs, ML preprocessing scripts, and thumbnail generators everywhere.

In [ ]:
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance

_FILTERS = {
    'blur':         ImageFilter.BLUR,
    'sharpen':      ImageFilter.SHARPEN,
    'edge_enhance': ImageFilter.EDGE_ENHANCE,
}

_src = Image.new('RGB', (200, 150), color=(80, 120, 180))


## Task

Implement `image_pipeline(img, steps) -> Image.Image`:

- Start with `result = img.copy()` (don't modify the original)
- Iterate `steps`, each a `(op_name, params_dict)` tuple
- Apply each operation to `result`
- Raise `ValueError` for unknown ops

| Op | params | Operation |
|----|--------|-----------|
| `'resize'` | `{width, height}` | `resize` with LANCZOS |
| `'crop'` | `{width, height}` | center crop |
| `'grayscale'` | `{}` | convert to L mode |
| `'filter'` | `{name}` | look up in `_FILTERS` |
| `'brightness'` | `{factor}` | `ImageEnhance.Brightness` |
| `'contrast'` | `{factor}` | `ImageEnhance.Contrast` |

## Your Implementation

In [ ]:
def image_pipeline(img: Image.Image, steps: list) -> Image.Image:
    """Apply a sequence of image processing steps in order.

    Each step is a (operation_name, params_dict) tuple.

    Supported operations:
        ('resize',     {'width': int, 'height': int})
        ('crop',       {'width': int, 'height': int})   # center crop
        ('grayscale',  {})
        ('filter',     {'name': str})                   # key in _FILTERS
        ('brightness', {'factor': float})               # 1.0 = unchanged
        ('contrast',   {'factor': float})               # 1.0 = unchanged

    Args:
        img:   starting PIL Image (not modified)
        steps: list of (op_name, params) tuples
    Returns:
        Processed Image
    Raises:
        ValueError for unknown operations
    """
    raise NotImplementedError


In [ ]:
def image_pipeline(img: Image.Image, steps: list) -> Image.Image:
    result = img.copy()
    for op, params in steps:
        if op == 'resize':
            result = result.resize(
                (params['width'], params['height']), Image.Resampling.LANCZOS)
        elif op == 'crop':
            w, h = params['width'], params['height']
            iw, ih = result.size
            left, top = (iw - w) // 2, (ih - h) // 2
            result = result.crop((left, top, left + w, top + h))
        elif op == 'grayscale':
            result = result.convert('L')
        elif op == 'filter':
            f = _FILTERS.get(params['name'].lower())
            if f is None:
                raise ValueError(f"Unknown filter: {params['name']!r}")
            result = result.filter(f)
        elif op == 'brightness':
            result = ImageEnhance.Brightness(result).enhance(params['factor'])
        elif op == 'contrast':
            result = ImageEnhance.Contrast(result).enhance(params['factor'])
        else:
            raise ValueError(f"Unknown operation: {op!r}")
    return result


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # Empty pipeline — result identical to input
    result = image_pipeline(_src, [])
    assert list(result.getdata()) == list(_src.getdata()), (
        "Empty pipeline should return a copy with identical pixels")
    score += 1; print("\u2705 empty steps returns identical image")

    # Single resize step
    result = image_pipeline(_src, [('resize', {'width': 100, 'height': 75})])
    assert result.size == (100, 75), f"Expected (100, 75), got {result.size}"
    score += 1; print("\u2705 resize step works")

    # Multi-step: resize + grayscale
    result = image_pipeline(_src, [
        ('resize',    {'width': 60, 'height': 60}),
        ('grayscale', {}),
    ])
    assert result.size == (60, 60) and result.mode == 'L', (
        f"Expected (60, 60) L, got {result.size} {result.mode}")
    score += 1; print("\u2705 resize + grayscale pipeline works")

    # Brightness step changes pixel values
    import numpy as np
    bright = image_pipeline(_src, [('brightness', {'factor': 2.0})])
    arr_src   = np.array(_src, dtype=float)
    arr_bright = np.array(bright, dtype=float)
    assert arr_bright.mean() > arr_src.mean(), (
        "brightness factor=2.0 should increase mean pixel value")
    score += 1; print("\u2705 brightness step increases pixel values")

    # Unknown operation raises ValueError
    raised = False
    try:
        image_pipeline(_src, [('teleport', {})])
    except ValueError:
        raised = True
    assert raised, "Unknown op should raise ValueError"
    score += 1; print("\u2705 unknown operation raises ValueError")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def image_pipeline(img: Image.Image, steps: list) -> Image.Image:
    result = img.copy()
    for op, params in steps:
        if op == 'resize':
            result = result.resize(
                (params['width'], params['height']), Image.Resampling.LANCZOS)
        elif op == 'crop':
            w, h = params['width'], params['height']
            iw, ih = result.size
            left, top = (iw - w) // 2, (ih - h) // 2
            result = result.crop((left, top, left + w, top + h))
        elif op == 'grayscale':
            result = result.convert('L')
        elif op == 'filter':
            f = _FILTERS.get(params['name'].lower())
            if f is None:
                raise ValueError(f"Unknown filter: {params['name']!r}")
            result = result.filter(f)
        elif op == 'brightness':
            result = ImageEnhance.Brightness(result).enhance(params['factor'])
        elif op == 'contrast':
            result = ImageEnhance.Contrast(result).enhance(params['factor'])
        else:
            raise ValueError(f"Unknown operation: {op!r}")
    return result
```

**Why `img.copy()` at the start?** If `steps` is empty the caller gets back a copy, not the same object. This prevents accidental mutation of the original when the caller passes the result to another pipeline. The copy cost is negligible — it's the safe default.

</details>